In [1]:
import pandas as pd

fact = pd.read_csv("../data/processed/fact_orders.csv")

print(fact.shape)
fact.head()

(99441, 16)


,order_id,customer_id,customer_city,customer_state,order_status,order_purchase_timestamp,delivery_days,delay_days,is_late,on_time_delivery,payment_value,review_score,total_price,freight_value,order_value,item_count
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,sao paulo,SP,delivered,2017-10-02 10:56:33,8.0,-8.0,0,1,38.71,4.0,29.99,8.72,38.71,1.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,barreiras,BA,delivered,2018-07-24 20:41:37,13.0,-6.0,0,1,141.46,4.0,118.70,22.76,141.46,1.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,vianopolis,GO,delivered,2018-08-08 08:38:49,9.0,-18.0,0,1,179.12,5.0,159.90,19.22,179.12,1.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,sao goncalo do amarante,RN,delivered,2017-11-18 19:28:06,13.0,-13.0,0,1,72.20,5.0,45.00,27.20,72.20,1.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,santo andre,SP,delivered,2018-02-13 21:18:39,2.0,-10.0,0,1,28.62,5.0,19.90,8.72,28.62,1.0


In [2]:
state_revenue = (
    fact.groupby("customer_state")
    .agg(
        revenue=("order_value","sum"),
        orders=("order_id","count")
    )
    .sort_values("revenue", ascending=False)
)

state_revenue.head(10)

,revenue,orders
customer_state,,
SP,5921678.12,41746
RJ,2129681.98,12852
MG,1856161.49,11635
RS,885826.76,5466
PR,800935.44,5045
BA,611506.67,3380
SC,610213.60,3637
DF,353229.44,2140
GO,347706.93,2020


In [3]:
state_revenue["revenue_per_order"] = (
    state_revenue["revenue"] /
    state_revenue["orders"]
)

state_revenue.sort_values(
    "revenue_per_order",
    ascending=False
).head(10)

,revenue,orders,revenue_per_order
customer_state,,,
PB,140987.81,536,263.036959
AC,19669.70,81,242.835802
AP,16262.80,68,239.158824
AL,96229.40,413,233.000969
RO,57558.02,253,227.502055
PA,217647.11,975,223.227805
TO,61354.42,280,219.122929
RR,10064.62,46,218.796087
PI,108132.28,495,218.449051


In [4]:
delivery_perf = (
    fact.groupby("customer_state")
    .agg(
        avg_delivery_days=("delivery_days","mean"),
        late_rate=("is_late","mean")
    )
)

delivery_perf["late_rate"] *= 100

delivery_perf.sort_values(
    "late_rate",
    ascending=False
).head(10)

,avg_delivery_days,late_rate
customer_state,,
AL,24.040302,20.581114
MA,21.117155,16.733601
SE,21.029851,14.571429
PI,18.993697,13.333333
CE,20.817826,13.173653
BA,18.866400,11.715976
RJ,14.849186,11.632431
PA,23.316068,10.871795
RR,28.975610,10.869565


In [5]:
customer_sat = (
    fact.groupby("customer_state")
    .agg(
        avg_review=("review_score","mean"),
        avg_delivery=("delivery_days","mean")
    )
)

customer_sat.sort_values(
    "avg_review",
    ascending=False
).head(10)

,avg_review,avg_delivery
customer_state,,
AP,4.176471,26.731343
AM,4.175676,25.986207
PR,4.168682,11.526711
SP,4.160730,8.298061
RS,4.126235,14.819237
MG,4.120541,11.543813
MS,4.106294,15.191155
TO,4.100000,17.226277
MT,4.086549,17.593679


In [6]:
fact["delivery_bucket"] = pd.cut(
    fact["delivery_days"],
    bins=[0,5,10,20,500],
    labels=[
        "Fast",
        "Normal",
        "Slow",
        "Very Slow"
    ]
)

fact.groupby(
    "delivery_bucket"
)["review_score"].mean()

C:\Users\rishu\AppData\Local\Temp\ipykernel_23132\1879187777.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  fact.groupby(


delivery_bucket
Fast         4.422938
Normal       4.337178
Slow         4.182018
Very Slow    3.099211
Name: review_score, dtype: float64

In [7]:
fact.groupby(
    "customer_state"
)["freight_value"].mean().sort_values(
    ascending=False
).head(10)

customer_state
RR    48.591087
PB    48.345357
RO    46.224211
AC    45.515432
PI    43.038945
MA    42.599689
TO    42.052616
AP    41.007353
SE    40.902812
PA    39.896186
Name: freight_value, dtype: float64

In [8]:
fact.groupby(
    "customer_city"
)["order_id"].count().sort_values(
    ascending=False
).head(20)

customer_city
sao paulo                15540
rio de janeiro            6882
belo horizonte            2773
brasilia                  2131
curitiba                  1521
campinas                  1444
porto alegre              1379
salvador                  1245
guarulhos                 1189
sao bernardo do campo      938
niteroi                    849
santo andre                797
osasco                     746
santos                     713
goiania                    692
sao jose dos campos        691
fortaleza                  654
sorocaba                   633
recife                     613
florianopolis              570
Name: order_id, dtype: int64

In [9]:
fact[
[
"delivery_days",
"delay_days",
"review_score",
"payment_value",
"freight_value",
"order_value"
]
].corr()

,delivery_days,delay_days,review_score,payment_value,freight_value,order_value
delivery_days,1.000000,0.607716,-0.335178,0.069496,0.167077,0.069458
delay_days,0.607716,1.000000,-0.269171,-0.017743,-0.049303,-0.017702
review_score,-0.335178,-0.269171,1.000000,-0.050246,-0.089257,-0.047669
payment_value,0.069496,-0.017743,-0.050246,1.000000,0.492681,0.999987
freight_value,0.167077,-0.049303,-0.089257,0.492681,1.000000,0.492603
order_value,0.069458,-0.017702,-0.047669,0.999987,0.492603,1.000000


In [10]:
state_revenue["revenue_per_order"] = (
    state_revenue["revenue"] /
    state_revenue["orders"]
)

state_revenue.sort_values(
    "revenue_per_order",
    ascending=False
).head(10)

,revenue,orders,revenue_per_order
customer_state,,,
PB,140987.81,536,263.036959
AC,19669.70,81,242.835802
AP,16262.80,68,239.158824
AL,96229.40,413,233.000969
RO,57558.02,253,227.502055
PA,217647.11,975,223.227805
TO,61354.42,280,219.122929
RR,10064.62,46,218.796087
PI,108132.28,495,218.449051


In [11]:
fact["delivery_bucket"] = pd.cut(
    fact["delivery_days"],
    bins=[0,5,10,20,500],
    labels=[
        "Fast",
        "Normal",
        "Slow",
        "Very Slow"
    ]
)

fact.groupby(
    "delivery_bucket"
)["review_score"].mean()

C:\Users\rishu\AppData\Local\Temp\ipykernel_23132\1879187777.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  fact.groupby(


delivery_bucket
Fast         4.422938
Normal       4.337178
Slow         4.182018
Very Slow    3.099211
Name: review_score, dtype: float64